# Relocalisation des tracks Rekordbox

Ce notebook documente le script `worker/relocate_tracks.py`.

**Objectif** : vérifier que chaque track actif est physiquement dans le bon dossier `W_MIX`,
déplacer les fichiers mal placés, et mettre à jour le pointeur `OrgFolderPath` dans Rekordbox.

**Règles** :
- Tag style (ex: `Tech House`) → `C:\Users\willi\Music\W_MIX\W - Tech House\`
- Tag `SoundCloud` → `C:\Users\willi\Music\W_MIX\_IMPORT\`
- Tags utilitaires (`TO_*`) → ignorés
- Pas de tag style → signalé, aucune action

## Connexion & données

In [ ]:
from collections import defaultdict
from pyrekordbox import Rekordbox6Database

db = Rekordbox6Database()
tracks = [t for t in db.get_content() if t.rb_data_status == 256]
print(f"{len(tracks)} tracks actifs")

## Structure des tags

Les tags sont organisés en groupes dans Rekordbox. Seul le groupe **STYLE** contient les tags
qui déterminent le dossier cible. Les tags `TO_*` (groupe PROCESS) sont des tags de workflow,
le groupe SOURCE distingue les fichiers SoundCloud.

In [ ]:
tags = list(db.get_my_tag())
tag_songs = list(db.get_my_tag_songs())

used_tags = {s.MyTag.Name for s in tag_songs}
groups = {t.ID: t.Name for t in tags if t.Attribute == 1}

result = {name: [] for name in groups.values()}
for tag in tags:
    if tag.Attribute == 0 and tag.ParentID in groups and tag.Name in used_tags:
        result[groups[tag.ParentID]].append(tag.Name)

{k: v for k, v in result.items() if v}

## Logique de filtrage des tags style

Un tag est considéré "style" s'il :
- Ne commence **pas** par `TO_`
- N'est **pas** `soundcloud`

Certains tags ont un nom qui diffère du nom de dossier sur disque (le `/` devient `_`, le `-` devient espace) :
une table de correspondance gère ces cas.

In [ ]:
import os

W_MIX_ROOT = r"C:\Users\willi\Music\W_MIX"

TAG_TO_FOLDER = {
    "Classic/Min. Techno": "W - Classic_Min. Techno",
    "Hard/Dark Techno":    "W - Hard_Dark Techno",
    "Nu-Disco":            "W - Nu Disco",
}

def _is_style_tag(tag: str) -> bool:
    return not tag.upper().startswith("TO_") and tag.lower() != "soundcloud"

def get_style_tag(tag_names):
    return next((t for t in tag_names if _is_style_tag(t)), None)

def expected_folder(tag_names):
    if "soundcloud" in [t.lower() for t in tag_names]:
        return os.path.join(W_MIX_ROOT, "_IMPORT")
    style = get_style_tag(tag_names)
    if style:
        folder_name = TAG_TO_FOLDER.get(style, f"W - {style}")
        return os.path.join(W_MIX_ROOT, folder_name)
    return None

# Vérification sur quelques exemples
examples = [
    ["TO_ENERGY", "TO_CUE", "Tech House"],
    ["TO_RATE", "SoundCloud", "TO_ENERGY"],
    ["Classic/Min. Techno", "TO_LOOP"],
    ["Nu-Disco"],
    ["TO_ENERGY", "TO_SORT"],
]

for tags_ex in examples:
    style = get_style_tag(tags_ex)
    folder = expected_folder(tags_ex)
    print(f"{tags_ex}")
    print(f"  style={style!r}  →  {folder}\n")

## Audit — état actuel de la bibliothèque

On parcourt tous les tracks actifs et on compare `OrgFolderPath` avec le dossier attendu.

In [ ]:
ok = []
to_move = []
no_tag = []

for track in tracks:
    tag_names = list(track.MyTagNames or [])
    current_path = track.OrgFolderPath or track.FolderPath

    target_folder = expected_folder(tag_names)
    if target_folder is None:
        no_tag.append(track)
        continue

    filename = os.path.basename(current_path)
    target_path = os.path.join(target_folder, filename)

    if os.path.normcase(current_path) == os.path.normcase(target_path):
        ok.append(track)
    else:
        to_move.append((track, current_path, target_path))

print(f"En place    : {len(ok)}")
print(f"A déplacer  : {len(to_move)}")
print(f"Sans tag    : {len(no_tag)}")

In [ ]:
# Détail des tracks sans tag style
print("Tracks sans tag style :")
for t in no_tag:
    print(f"  {t.ArtistName} - {t.Title}  |  tags={list(t.MyTagNames or [])}")

In [ ]:
# Détail des tracks à déplacer (si il y en a)
for track, src, dst in to_move:
    print(f"{track.ArtistName} - {track.Title}")
    print(f"  {src}")
    print(f"  -> {dst}\n")

if not to_move:
    print("Tout est en place.")

## Répartition par style

In [ ]:
by_style = defaultdict(list)
for track in tracks:
    tag_names = list(track.MyTagNames or [])
    if "soundcloud" in [t.lower() for t in tag_names]:
        by_style["_IMPORT (SoundCloud)"].append(track)
    else:
        style = get_style_tag(tag_names)
        by_style[style or "(aucun tag)"].append(track)

for style, t_list in sorted(by_style.items(), key=lambda x: -len(x[1])):
    print(f"{len(t_list):>4}  {style}")

## Utilisation du script

```bash
# Vérifier sans toucher à rien
python worker/relocate_tracks.py --dry-run

# Déplacer les fichiers + mettre à jour Rekordbox
python worker/relocate_tracks.py
```

Le script :
1. Lit `OrgFolderPath` (chemin Windows réel) pour chaque track
2. Calcule le dossier attendu selon le tag style
3. Déplace le fichier avec `shutil.move`
4. Met à jour `OrgFolderPath` **et** `FolderPath` dans la DB Rekordbox
5. Commit en une seule transaction à la fin

> **Note** : Rekordbox doit être fermé pendant l'exécution du script pour éviter les conflits d'accès à la base SQLite.